# Stage 11 - the reranker under an RL objective

Design: `docs/stage11_reranker_rl.md`.

## Setup

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/RAG chunk optimize'

from google.colab import drive
drive.mount('/content/drive')

import os, sys
if not os.path.isfile(os.path.join(PROJECT_DIR, 'config.py')):
    raise RuntimeError(f'Missing config.py under {PROJECT_DIR!r}.')

os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

import config as C
C.ensure_dirs()
print('project:', PROJECT_DIR)
print(C.summary())

## Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## Training groups

In [ ]:
!python scripts/26_build_stage11_data.py

## Training smoke check

In [ ]:
!python scripts/27_train_reranker_rl.py --smoke

## Train both arms

In [ ]:
!python scripts/27_train_reranker_rl.py

## Dev gate

In [ ]:
!python scripts/28_eval_reranker_rl.py --dev

## Gate

In [ ]:
import json, os, pathlib

latest = pathlib.Path(os.environ['RAG_DATA_ROOT']) / 'results' / 'latest'
gate_path = latest / 'stage11_dev_gate.json'
gate = json.loads(gate_path.read_text(encoding='utf-8')) if gate_path.exists() else None
GO = bool(gate) and gate['verdict'] == 'GO'
if gate is None:
    print('no dev gate result - the dev evaluation did not finish.')
else:
    print(f"dev gate: {gate['verdict']} - {gate['why']}")
    print(f"RL - CE dev R@1 {gate['rl_minus_ce_r1']:+.4f}, 95% CI "
          f"[{gate['ci95'][0]:+.4f}, {gate['ci95'][1]:+.4f}], "
          f"RL live fraction {gate['live_fraction']:.3f}")
print('\nGO - the final evaluation will run.' if GO else
      '\nNO-GO - the final evaluation is skipped. Record the NO-GO as the result.')

## Stage 6 evaluation (GO required)

In [ ]:
if GO:
    !python scripts/28_eval_reranker_rl.py
else:
    print('skipped: the dev gate did not pass.')

## Optional fixed 6/0 evaluation

In [ ]:
RUN_SECONDARY = False   # fixed 6/0 is optional

if GO and RUN_SECONDARY:
    !python scripts/28_eval_reranker_rl.py --configs 15:0,6:0
else:
    print('secondary config skipped.')

## Review

In [ ]:
from IPython.display import Image, Markdown, display

summary = latest / 'stage11_summary.md'
if summary.exists():
    display(Markdown(summary.read_text(encoding='utf-8')))
    png = latest / 'stage11_delta.png'
    if png.exists():
        display(Image(str(png)))
else:
    print('no Stage 11 summary - the final evaluation has not run.')